# Lesson 07 - திட்டமிடல் வடிவமைப்பு முறை

இந்த نوட்புக் Microsoft Agent Framework பயன்படுத்தி AI முகவர்கள் க்கான **திட்டமிடல் வடிவமைப்பு முறை** ஐ விளக்குகிறது.
நீங்கள் எப்படி சிக்கலான பயண கோரிக்கைகளை கட்டமைக்கப்பட்ட துணைபணிகளாகப் பிரிப்பது, அவற்றை நிபுணர் முகவர்களுக்கு ஒப்படைப்பது,
மற்றும் உருவாக்கப்பட்ட திட்டத்தை நிறைவேற்றுவது — எல்லாம் Pydantic மாதிரிகளால் இயக்கப்படும் கட்டமைக்கப்பட்ட வெளிப்பாட்டை பயன்படுத்தி கற்றுக்கொள்வீர்கள்.


## அமைப்பு


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## பணி பரிந்துரை

பணி பரிந்துரை திட்டமிடல் வடிவமைப்பு முறைமையின் மையமாகும். ஒரு எளிய முகவரிடம் முழுமையான, கடுமையடைந்த வேண்டுகோள் ஒன்றைப் பெறுவதற்குப் பதிலாக, நாம் பிரச்சினையை சிறிய, நன்கு வரையறுக்கப்பட்ட **உபபணிகள்** ஆகப் பிரிக்கின்றோம். ஒவ்வொரு உபபணிக்கும் தெளிவான முன்னுரிமைகள் மற்றும் தாக்கிறப்பு வரிசைப்படி ஒரு நிபுண முகவருக்கு (எ.கா., விமானங்கள், ஓட்டல்கள், செயல்பாடுகள்) ஒதுக்கப்படுகிறது.

இந்த அணுகுமுறை பல நன்மைகளை அளிக்கிறது:
- **தெளிவுகள்**: ஒவ்வொரு உபபணி ஒரே பொறுப்பைக் கொண்டுள்ளது.
- **சமநிலை செயல்பாடு**: சுதந்திரமான உபபணிகள் ஒரே நேரத்தில் இயங்கலாம்.
- **நம்பகத்தன்மை**: தோல்விகள் தனி உபபணிகளுக்கு மட்டுமே வரையறுக்கப்படுகின்றன.
- **பட்ஜெட் கண்காணிப்பு**: செலவுகள் ஒவ்வொரு உபபணிக்கும் கணிக்கப்பட்டு மேலோட்டமாகச் சேர்க்கப்படுகின்றன.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## கட்டமைக்கப்பட்ட வெளியீட்டுடன் ஒரு திட்டமிடல் முகவரியை உருவாக்குதல்

திட்டமிடல் முகவர் **முன்னணி மேசை ஒருங்கிணைப்பாளராக** செயல்படுகிறது. ஒரு உயர்நிலை பயண கோரிக்கையை வழங்குவதால்
அது ஒரு கட்டமைக்கப்பட்ட `TravelPlan` ஐ உருவாக்குகிறது — கோரிக்கையை துணை பணிகளாக பிரித்து, முன்னுரிமைகளை அமைத்து,
மற்றும் ஒரு கான்சியர்ஜ் அல்லது செயலாக்க அடுக்கு பணியை நிறைவேற்ற Dependency ஐ உறுதிசெய்தல்.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## நிபுணத்துவ கருவிகளுடன் ஒரு திட்டத்தை நிறைவேற்றுதல்

முதல் மேசை பிரதிநிதி அமைத்த திட்டத்தை தயாரித்தவுடன், **கான்சிஅர்ஜ் பிரதிநிதி** அதை நிறைவேற்றுகிறான்.  
ஒவ்வொரு நிபுணத்துவ கருவியும் ஒரு வகை துணைபணி(கள்)க்கு ( விமானங்கள், மருத்துவமனைகள், நடவடிக்கைகள்) கையாளுகிறது. கான்சிஅர்ஜ் திட்டத்தின் துணைப்பணிகளின் சார்புக்கு ஏற்ப தொடர்புகளை வரிசைப்படுத்தி ஒவ்வொன்றையும் பொருத்தமான கருவிக்கு அனுப்புகிறான்.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் AI முகவர்கள் க்கான **திட்டமிடல் வடிவமைப்பு முறை** பற்றி கற்றுக்கொண்டீர்கள்:

1. **பணியை உடைத்தல்** — முன் டெஸ்க்கு திட்டமிடல் முகவர் ஒரு சிக்கலான பயணக் கோரிக்கையை திட்டமிட்ட
   துணைப் பணிகளாகப் பைடான்டிக் மாதிரிகள் மூலம் உடைத்து, ஒவ்வொன்றையும் முன்னுரிமைகள்
   மற்றும் சார்மானங்களுடன் ஒரு சிறப்பு முகவருக்கு ஒதுக்குகிறார்.
2. **உறுப்படுத்தப்பட்ட வெளியீடு** — `response_format` வழங்குவதன் மூலம், முகவர் உரையாளர்
   வடிவமான `TravelPlan` பொருளை திரும்ப அனுப்பி, பின்னர் செயல்முறை நம்பகமானதாக மாறுகிறது.
3. **திட்ட செயலாக்கம்** — ஒரு கன்சிர்ஜ் முகவர் துணைப் பணிகள் மூலம் சிறப்பு கருவிகள்
   (`book_flight`, `reserve_hotel`, `book_activity`) பயன்படுத்தித் திட்டத்தை நிறைவேற்றி முடிவுகளை
   அறிக்கை செய்கிறார்.

இந்த முறை *என்ன செய்ய வேண்டும்* (திட்டமிடல்) மற்றும் *எப்படிச் செய்ய வேண்டும்* (செயலாக்கம்)
என்று வேறுபடுத்துவதன் மூலம் முகவர்களை modular, சோதிக்கக் கூடிய மற்றும் விரிவாக்கக்கூடியவையாக
செய்கிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**தயவுசெய்து கவனிக்கவும்**:
இந்த ஆவணம் [Co-op Translator](https://github.com/Azure/co-op-translator) என்ற செயற்கை நுண்ணறிவு மொழிபெயர்ப்பு சேவையை பயன்படுத்தி மொழிபெயர்க்கப்பட்டது. நமது முயற்சி துல்லியமாக இருப்பதாக இருந்தாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் உள்ளடக்கப்படலாம் என்பதைக் கருத்தில் கொள்ளவும். அதன் அசல் ஆவணம் அதன் தாய்மொழியில் உள்ளதுதான் அதிகாரபூர்வமான ஆதாரமாகக் கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு வல்லுநர் மனித மொழிபெயர்ப்பை பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பு பயன்படுத்தியதன் மூலம் ஏற்படும் எந்த தவறான புரிந்துகொள்ளல்கள் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பேற்கவில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
